# Panel series diagnostics

This notebook demonstrates `IntermittencyAnalyzer`, `SeasonalPeriodDetector`, and the combined `SeriesProfiler` using a Nixtla-style panel.

In [2]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyshift.series import (
    IntermittencyAnalyzer,
    SeasonalPeriodDetector,
    SeriesProfiler,
)

In [3]:
n = 84
steps = np.arange(n)
dates = pd.date_range("2025-01-01", periods=n, freq="D")
rng = np.random.default_rng(42)

smooth = 6 + 1.5 * np.sin(2 * np.pi * steps / 7) + 0.02 * steps
intermittent = np.where(steps % 3 == 0, 3 + rng.uniform(0, 2, n), 0.0)
lumpy = np.where(steps % 4 == 0, rng.lognormal(1.2, 0.9, n), 0.0)

df = pd.concat(
    [
        pd.DataFrame({"unique_id": uid, "ds": dates, "y": values})
        for uid, values in [
            ("smooth_weekly", smooth),
            ("intermittent", intermittent),
            ("lumpy", lumpy),
        ]
    ],
    ignore_index=True,
)
df.head()

,unique_id,ds,y
0,smooth_weekly,2025-01-01,6.000000
1,smooth_weekly,2025-01-02,7.192747
2,smooth_weekly,2025-01-03,7.502392
3,smooth_weekly,2025-01-04,6.710826
4,smooth_weekly,2025-01-05,5.429174


## Demand occurrence

`IntermittencyAnalyzer.profile()` returns the occurrence and magnitude diagnostics for each series. Detailed interval arrays remain available in `results_`.

In [4]:
intermittency = IntermittencyAnalyzer().fit(df)
intermittency.profile()

,unique_id,adi,cv2,zero_proportion,interval_cv,classification
0,intermittent,3.0,0.013614,0.666667,0.0,intermittent
1,lumpy,4.0,1.253091,0.750000,0.0,lumpy
2,smooth_weekly,1.0,0.027822,0.000000,0.0,smooth


## Candidate seasonal periods

The detector profile exposes only `candidate_periods`; FFT frequencies, power, and peaks can be inspected through `results_`.

In [5]:
seasonality = SeasonalPeriodDetector(top_k=2).fit(df)
seasonality.profile()

,unique_id,candidate_periods
0,intermittent,"[3, 6]"
1,lumpy,"[2, 4]"
2,smooth_weekly,[7]


## Combined series summary

`SeriesProfiler.summary()` combines demand occurrence, predictability, temporal structure, and spectral structure into one row per unique ID.

In [13]:
profiler = SeriesProfiler(top_k=2).fit(df)
profiler.summary()

,unique_id,adi,cv2,zero_prop,interval_cv,class,foreca,limit,hurst,trend_r2,trend_pvalue,spectral_conc,candidate_periods
0,intermittent,3.0,0.013614,0.666667,0.0,intermittent,0.954999,0.386935,0.305477,0.001274,0.747201,0.957805,"[3, 6]"
1,lumpy,4.0,1.253091,0.750000,0.0,lumpy,0.197007,0.419804,0.443839,0.008430,0.406184,0.074067,"[2, 4]"
2,smooth_weekly,1.0,0.027822,0.000000,0.0,smooth,0.992008,0.281646,0.501202,0.136375,0.000546,0.992513,[7]


In [16]:
# Inspect one semantic section without flattening the result.
profiler.results_["smooth_weekly"]["spectral_structure"]

{'spectral_conc': 0.9925125683723844, 'candidate_periods': [7]}